# VisionAI — YOLOv8 Fine-Tuning on Custom Video Frames

**Goal:** Fine-tune YOLOv8 on your own ground-level / night-time CCTV video frames  
so the model understands YOUR camera angles, lighting and scene.

## Pipeline
```
Your videos
   → extract frames (1 frame every 2 s)
   → auto-label with current best model (pseudo-labels)
   → split train/val
   → fine-tune YOLOv8s for 30 epochs
   → export best.pt
```

## Instructions
1. Upload your `.mp4` / `.avi` videos to Kaggle Dataset or attach them as input.
2. Set `VIDEO_DIR` below.
3. Run all cells top-to-bottom.
4. Download `runs/detect/finetune/weights/best.pt` and place it in `backend/models/`.


In [ ]:
# ── Install deps ──────────────────────────────────────────────────────────
!pip install -q ultralytics opencv-python-headless

In [ ]:
import os, cv2, shutil, random, yaml
from pathlib import Path
from ultralytics import YOLO

# ══ CONFIGURE THESE ══════════════════════════════════════════════════════════
VIDEO_DIR        = '/kaggle/input/your-dataset/'   # folder containing .mp4/.avi files
FRAME_INTERVAL_S = 2       # extract 1 frame every N seconds
PSEUDO_MODEL     = 'yolov8s.pt'  # pretrained model used for auto-labelling
                                  # (or upload your yolov8s_visdrone_best.pt here)
CONF_PSEUDO      = 0.25    # confidence threshold for pseudo-labels
TRAIN_SPLIT      = 0.85    # 85% train, 15% val
FINETUNE_EPOCHS  = 30
IMGSZ            = 640     # 640 for speed; 1280 for max accuracy (slower)
BATCH            = 16      # reduce to 8 if VRAM OOM
MODEL_TO_FINETUNE = 'yolov8s.pt'   # base model weights
DATASET_ROOT     = '/kaggle/working/visionai_dataset'
# ═════════════════════════════════════════════════════════════════════════════
print('Config loaded ✅')

In [ ]:
# ── Step 1: Extract frames from videos ───────────────────────────────────────
frames_dir = Path(DATASET_ROOT) / 'raw_frames'
frames_dir.mkdir(parents=True, exist_ok=True)

video_files = list(Path(VIDEO_DIR).glob('**/*.mp4')) + \
              list(Path(VIDEO_DIR).glob('**/*.avi')) + \
              list(Path(VIDEO_DIR).glob('**/*.mov'))

print(f'Found {len(video_files)} video(s)')
total_frames = 0

for vid_path in video_files:
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    interval = max(1, int(fps * FRAME_INTERVAL_S))
    frame_idx = 0
    saved = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % interval == 0:
            name = f"{vid_path.stem}_f{frame_idx:06d}.jpg"
            cv2.imwrite(str(frames_dir / name), frame)
            saved += 1
        frame_idx += 1
    cap.release()
    print(f'  {vid_path.name}: {saved} frames extracted')
    total_frames += saved

print(f'\nTotal frames extracted: {total_frames}')

In [ ]:
# ── Step 2: Auto-label with pseudo-label model ────────────────────────────────
print(f'Loading pseudo-label model: {PSEUDO_MODEL}')
pseudo_model = YOLO(PSEUDO_MODEL)

labels_dir = Path(DATASET_ROOT) / 'raw_labels'
labels_dir.mkdir(exist_ok=True)

frame_files = sorted(frames_dir.glob('*.jpg'))
print(f'Auto-labelling {len(frame_files)} frames at conf={CONF_PSEUDO}...')

labelled = 0
for img_path in frame_files:
    results = pseudo_model.predict(str(img_path), conf=CONF_PSEUDO, verbose=False)
    r = results[0]
    h, w = r.orig_shape

    label_lines = []
    for box in r.boxes:
        cls_id = int(box.cls[0])
        cx, cy, bw, bh = box.xywhn[0].tolist()  # normalized
        label_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    if label_lines:   # only keep frames that have at least 1 detection
        label_path = labels_dir / (img_path.stem + '.txt')
        label_path.write_text('\n'.join(label_lines))
        labelled += 1

print(f'Labelled {labelled}/{len(frame_files)} frames (skipped blank frames)')

In [ ]:
# ── Step 3: Build YOLO dataset structure ─────────────────────────────────────
# Only keep frames that were labelled
labelled_stems = {p.stem for p in labels_dir.glob('*.txt')}
labelled_imgs  = [p for p in frame_files if p.stem in labelled_stems]
random.shuffle(labelled_imgs)

split = int(len(labelled_imgs) * TRAIN_SPLIT)
splits = {'train': labelled_imgs[:split], 'val': labelled_imgs[split:]}
print(f'Train: {len(splits["train"])} | Val: {len(splits["val"])}')

for split_name, imgs in splits.items():
    img_dir = Path(DATASET_ROOT) / split_name / 'images'
    lbl_dir = Path(DATASET_ROOT) / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img_path in imgs:
        shutil.copy(img_path, img_dir / img_path.name)
        lbl_path = labels_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            shutil.copy(lbl_path, lbl_dir / lbl_path.name)

# Get class names from pseudo model
class_names = list(pseudo_model.names.values())

# Write dataset YAML
data_yaml = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'val/images',
    'nc':    len(class_names),
    'names': class_names,
}
yaml_path = Path(DATASET_ROOT) / 'dataset.yaml'
yaml_path.write_text(yaml.dump(data_yaml))
print(f'Dataset YAML written: {yaml_path}')
print(f'Classes ({len(class_names)}): {class_names}')

In [ ]:
# ── Step 4: Fine-tune YOLOv8 ─────────────────────────────────────────────────
print(f'Fine-tuning {MODEL_TO_FINETUNE} for {FINETUNE_EPOCHS} epochs at imgsz={IMGSZ}...')
model = YOLO(MODEL_TO_FINETUNE)

results = model.train(
    data=str(yaml_path),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,          # early stopping
    augment=True,         # mosaic, flips, HSV jitter — good for night robustness
    hsv_v=0.6,            # brightness augmentation range (helps night↔day transfer)
    degrees=5,            # slight rotation (CCTV angle variation)
    project='/kaggle/working/runs/detect',
    name='finetune',
    exist_ok=True,
)
print('Training complete!')
print(f'Best model: /kaggle/working/runs/detect/finetune/weights/best.pt')

In [ ]:
# ── Step 5: Evaluate fine-tuned model on val split ───────────────────────────
best_pt = '/kaggle/working/runs/detect/finetune/weights/best.pt'
finetuned = YOLO(best_pt)
val_results = finetuned.val(data=str(yaml_path), imgsz=IMGSZ, verbose=True)

print('\n── Validation Metrics ──────────────────────────────')
print(f'mAP@0.5:      {val_results.box.map50:.4f}  ({val_results.box.map50*100:.2f}%)')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}  ({val_results.box.map*100:.2f}%)')
print(f'Precision:    {val_results.box.mp:.4f}')
print(f'Recall:       {val_results.box.mr:.4f}')
print('────────────────────────────────────────────────────')
print('\n✅ Download best.pt from Output → Files → runs/detect/finetune/weights/best.pt')
print('   Then place it in backend/models/ and select it from the ModelSelector.')

In [ ]:
# ── Step 6 (Optional): Per-lighting evaluation ───────────────────────────────
# Evaluate model separately on night vs day frames
import numpy as np

val_imgs = list((Path(DATASET_ROOT) / 'val' / 'images').glob('*.jpg'))
night_imgs = []
day_imgs   = []

for img_path in val_imgs:
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        brightness = img.mean()
        if brightness < 60:
            night_imgs.append(str(img_path))
        else:
            day_imgs.append(str(img_path))

print(f'Night frames: {len(night_imgs)} | Day/dusk frames: {len(day_imgs)}')

def quick_eval(model, img_list, label='subset', conf=0.25):
    if not img_list:
        print(f'{label}: no images')
        return
    total_det = 0
    for p in img_list[:50]:   # sample 50
        r = model.predict(p, conf=conf, verbose=False)
        total_det += len(r[0].boxes)
    print(f'{label} avg detections/frame: {total_det/min(50,len(img_list)):.1f}')

quick_eval(finetuned, night_imgs, 'Night')
quick_eval(finetuned, day_imgs,   'Day/Dusk')